# Cross-registrate sessions (python 3.8.15)

Specify directories and dataset patterns

In [103]:
import os
import sys
import warnings
import itertools as itt
import numpy as np
import xarray as xr
import holoviews as hv
import pandas as pd
from holoviews.operation.datashader import datashade, regrid
from dask.diagnostics import ProgressBar
from pathlib import Path
from IPython.display import display
from ipyfilechooser import FileChooser
import param
import panel as pn
import warnings
from holoviews import opts
import ipywidgets as widgets
import cv2
import json
import re
import math
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
import pickle
from matplotlib.widgets import TextBox
from matplotlib.lines import Line2D
pn.extension()
hv.extension("bokeh")

In [104]:
minian_path= "/home/thea.michel/minian"
sys.path.append(minian_path)
from minian.cross_registration import (calculate_centroids, calculate_centroid_distance, calculate_mapping,
                                       group_by_session, resolve_mapping, fill_mapping)
from minian.motion_correction import estimate_motion, apply_transform
from minian.utilities import open_minian, open_minian_mf
from minian.visualization import AlignViewer

minian_path = "."

f_pattern = r"minian"
id_dims = ["session"]

output_size = 100

#hv.notebook_extension('bokeh', width=100)
pbar = ProgressBar(minimum=2)
pbar.register()

In [105]:
try:
    %store -r dpath
except:
    print("data not in strore")
    dpath ="//10.69.168.1/crnldata/forgetting/"


# Set up Initial Basic Parameters#
minian_path = "."

fc1 = FileChooser(dpath,select_default=True, show_only_dirs = True, title = "<b>Choose Mice Folder</b>", layout=widgets.Layout(width='100%'))
display(fc1)

# Sample callback function
def update_my_folder(chooser):
    global dpath
    dpath = chooser.selected
    %store dpath
    return 

# Register callback function
fc1.register_callback(update_my_folder)

FileChooser(path='/crnldata/forgetting/Aurelie/MiniscopeOE_data/L1NDNF_mice/NV/Allocentric_task', filename='',…

In [106]:
DATE_RE = re.compile(r"^\d{4}[-_]\d{2}[-_]\d{2}$")   
ANIMAL_ROOT = "L1NDNF_mice"                           

def get_animal_id(minian_dir, dpath=None):
    parts = Path(minian_dir).parts
    if ANIMAL_ROOT in parts:
        return parts[parts.index(ANIMAL_ROOT) + 1]
    if dpath is not None:                             
        return Path(dpath).name
    raise ValueError(f"'{ANIMAL_ROOT}' introuvable dans : {minian_dir}")

def get_session_id(minian_dir):
    """minian_dir = dossier .../<date>/.../<heure>/My_V4_Miniscope/minian  ->  'date_heure'"""
    p = Path(minian_dir)
    # date : dossier-date le plus proche au-dessus de l'heure (quel que soit le nombre de sous-dossiers)
    date_folder = next((x for x in reversed(p.parts) if DATE_RE.match(x)), None)
    if date_folder is None:
        raise ValueError(f"Aucun dossier de type date dans : {minian_dir}")
    date = date_folder.replace("-", "_")
    time = p.parents[1].name                          # parents[0] = My_V4_Miniscope
    return f"{date}_{time}"

Try to load a pre-existing mappping

In [107]:
try: 
    shiftds=xr.open_dataset(os.path.join(dpath, "shiftds_Oct10.nc"))
    temps=shiftds.temps
    shifts=shiftds.shifts
    temps_sh=shiftds.temps_shifted
    with open(os.path.join(dpath, "cents_Oct10.pkl"), "rb") as f:
        cents = pickle.load(f)
    print('Cross-registration already done !')
    try: 
        with open(os.path.join(dpath, "mappings_Oct10_edited.pkl"), "rb") as f:
            mappings_meta_fill = pickle.load(f)
            print('... Loading the edited cross-registration')
    except: 
        with open(os.path.join(dpath, "mappings_Oct10.pkl"), "rb") as f:
            mappings_meta_fill = pickle.load(f)     
            print('... Loading the original cross-registration')


    drop_units = {}
    root = Path(dpath)
    files = sorted(Path(root).glob("**/minian/TodropFileAB.json"))
    for TodropFile in files:
        with open(TodropFile) as fp:
            units = json.load(fp)
        if len(Path(TodropFile).parents) > 13:# and "Habituation" not in str(TodropFile):
            session = (Path(TodropFile).parents[5].name + "_" + Path(TodropFile).parents[2].name)
        else: 
            session = (Path(TodropFile).parents[4].name + "_" + Path(TodropFile).parents[2].name)
        drop_units.setdefault(session, []).extend(units)
        #print(f"{len(units)} units to drop for {session}")
        #print(units)
        
    # Remove dropped units (AB modified => works much better!)
    cents_unittokeep = cents.copy()
    for session, cell_nums in drop_units.items():
        cents_unittokeep = cents_unittokeep[~((cents_unittokeep['session'] == session) &(cents_unittokeep['unit_id'].isin(cell_nums)))]

except: 
    pass 

## Allign videos automatically

Open datasets

In [ ]:
# old not to do 
def correct_meta(ds,  minian_ds_path_Origin):
    path_obj =Path(minian_ds_path_Origin)
    ds['animal'] = path_obj.parts[6]
    if len(path_obj.parts) > 13:# and "Habituation" not in path_obj.parts:
        ds['session'] = path_obj.parents[4].name + '_' + path_obj.parents[1].name
    else: 
        ds['session'] = path_obj.parents[3].name + '_' + path_obj.parents[1].name
    print(f"['animal: {ds['animal'].values}']")
    return ds

minian_ds = open_minian_mf(dpath, id_dims, pattern=f_pattern, post_process=correct_meta)

In [108]:
#new to do. find date and time independently of the subfolder number 
def correct_meta(ds,  minian_ds_path_Origin):
    ds['animal'] = get_animal_id(minian_ds_path_Origin, dpath)
    ds['session'] = get_session_id(minian_ds_path_Origin)
    print(f"['animal: {ds['animal'].values} | session: {ds['session'].values}']")
    return ds

minian_ds = open_minian_mf(dpath, id_dims, pattern=f_pattern, post_process=correct_meta)

sessions = minian_ds.session.values
print(len(sessions), "sessions")
print(sessions)
print("doublons :", len(sessions) - len(set(sessions)))

opening dataset under /crnldata/forgetting/Aurelie/MiniscopeOE_data/L1NDNF_mice/NB/Allocentric_task/Probe/2026_02_20/Cheeseboard/12_03_38/My_V4_Miniscope
opening minian
['animal: NB | session: 2026_02_20_12_03_38']
['session: 2026_02_20_12_03_38']
opening dataset under /crnldata/forgetting/Aurelie/MiniscopeOE_data/L1NDNF_mice/NB/Allocentric_task/Probe/2026_02_20/SleepBefore/11_26_40/My_V4_Miniscope
opening minian
['animal: NB | session: 2026_02_20_11_26_40']
['session: 2026_02_20_11_26_40']
opening dataset under /crnldata/forgetting/Aurelie/MiniscopeOE_data/L1NDNF_mice/NB/Allocentric_task/Probe/2026_02_20/SleepBefore/10_56_15/My_V4_Miniscope
opening minian
['animal: NB | session: 2026_02_20_10_56_15']
['session: 2026_02_20_10_56_15']
opening dataset under /crnldata/forgetting/Aurelie/MiniscopeOE_data/L1NDNF_mice/NB/Allocentric_task/Training/2026_02_19/SleepAfter/13-17-36_13-32-57/13_17_38/My_V4_Miniscope
opening minian
['animal: NB | session: 2026_02_19_13_17_38']
['session: 2026_02_19

Remove dropped units from ROIs

In [ ]:
# old not to do 
drop_units = {}
root = Path(dpath)
files = sorted(Path(root).glob("**/minian/TodropFileAB.json"))
for TodropFile in files:
    with open(TodropFile) as fp:
        units = json.load(fp)
    if len(Path(TodropFile).parents) > 13:# and "Habituation" not in str(TodropFile) :
        session = (Path(TodropFile).parents[5].name + "_" + Path(TodropFile).parents[2].name)
    else: 
        session = (Path(TodropFile).parents[4].name + "_" + Path(TodropFile).parents[2].name)
    drop_units.setdefault(session, []).extend(units)
    print(f"{len(units)} units to drop for {session}")

parts = []
for sess in minian_ds.session.values:
    A = minian_ds["A"].sel(session=sess)
    if sess in drop_units:
        keep = ~A.unit_id.isin(drop_units[sess])
        A = A.sel(unit_id=keep)
    parts.append(A)
A = xr.concat(parts, dim="session")

In [109]:
# new to do 
drop_units = {}
root = Path(dpath)
files = sorted(Path(root).glob("**/minian/TodropFileAB.json"))
for TodropFile in files:
    with open(TodropFile) as fp:
        units = json.load(fp)
    session = get_session_id(Path(TodropFile).parent)
    drop_units.setdefault(session, []).extend(units)
    print(f"{len(units)} units to drop for {session}")

parts = []
for sess in minian_ds.session.values:
    A = minian_ds["A"].sel(session=sess)
    if sess in drop_units:
        keep = ~A.unit_id.isin(drop_units[sess])
        A = A.sel(unit_id=keep)
    parts.append(A)
A = xr.concat(parts, dim="session")

12 units to drop for 2026_02_20_12_03_38
66 units to drop for 2026_02_20_10_56_15
75 units to drop for 2026_02_20_11_26_40
53 units to drop for 2026_02_16_13_01_31
63 units to drop for 2026_02_16_13_06_16
31 units to drop for 2026_02_16_13_11_15
55 units to drop for 2026_02_16_13_17_51
27 units to drop for 2026_02_16_13_25_00
25 units to drop for 2026_02_16_13_29_51
24 units to drop for 2026_02_16_13_34_03
39 units to drop for 2026_02_16_13_39_04
64 units to drop for 2026_02_16_14_28_33
56 units to drop for 2026_02_16_15_06_42
62 units to drop for 2026_02_16_11_35_10
77 units to drop for 2026_02_16_12_10_41
33 units to drop for 2026_02_17_12_06_50
36 units to drop for 2026_02_17_12_12_28
25 units to drop for 2026_02_17_12_16_31
26 units to drop for 2026_02_17_12_21_23
26 units to drop for 2026_02_17_12_26_26
34 units to drop for 2026_02_17_12_34_27
89 units to drop for 2026_02_17_12_40_08
64 units to drop for 2026_02_17_12_45_13
50 units to drop for 2026_02_17_13_11_21
45 units to drop

Option #1: Estimate shifts using raw images & minian function

In [110]:
temps = minian_ds['max_proj'].rename('temps')
#temps = temps.drop_sel(session=['session2_1', 'session2_2', 'session2_1', 'session3_1', 'session3_2', 'session3_3', 'session3_4'])
shifts = estimate_motion(temps, dim='session').compute().rename('shifts')
shifts_sorted = shifts.sortby("session")
for i, (session, shift) in enumerate(zip(shifts_sorted.session.values, shifts_sorted.values)):
    print(f"Session #{i}, {session}: {shift}")

[########################################] | 100% Completed | 2.62 s
[########################################] | 100% Completed | 2.72 s
Session #0, 2026_02_16_11_35_10: [-14.295   2.35 ]
Session #1, 2026_02_16_12_10_41: [-14.275   2.21 ]
Session #2, 2026_02_16_13_01_31: [-10.97166667  10.60666667]
Session #3, 2026_02_16_13_06_16: [-9.28166667 12.41333333]
Session #4, 2026_02_16_13_11_15: [-13.07833333  10.43333333]
Session #5, 2026_02_16_13_17_51: [-13.82833333  10.67333333]
Session #6, 2026_02_16_13_25_00: [-13.12833333  10.79333333]
Session #7, 2026_02_16_13_29_51: [-15.765  10.82 ]
Session #8, 2026_02_16_13_34_03: [-12.30166667  11.24666667]
Session #9, 2026_02_16_13_39_04: [121.62833333 -93.90333333]
Session #10, 2026_02_16_14_28_33: [-11.43166667  11.76333333]
Session #11, 2026_02_16_15_06_42: [-11.28166667  11.74333333]
Session #12, 2026_02_17_11_04_37: [-15.37833333   2.45333333]
Session #13, 2026_02_17_11_36_27: [-15.36833333   2.33333333]
Session #14, 2026_02_17_12_06_50: [1

Option #2: Estimate shifts using ROIs & AI function 

In [ ]:
def compute_shifts(images, max_shift=20, threshold=1):
    """
    Estimate translational shifts relative to the first session.
    Parameters
    ----------
    images : xr.DataArray
        (session, height, width)
    max_shift : int
        Maximum expected shift (pixels).
    threshold : float
        Fraction of the image maximum below which values are set to zero.
    Returns
    -------
    xr.DataArray
        (session, shift_dim)
    """
    ref = np.asarray(images[11], dtype=np.float32)
    # Ignore weak background
    ref = ref.copy()
    ref[ref < threshold * ref.max()] = 0
    ref_norm = np.linalg.norm(ref)
    shifts = []
    for i in range(images.shape[0]):
        img = np.asarray(images[i], dtype=np.float32)
        img = img.copy()
        img[img < threshold * img.max()] = 0
        img_norm = np.linalg.norm(img)
        best_score = -np.inf
        best_dx = 0
        best_dy = 0
        for dy in range(-max_shift, max_shift + 1):
            for dx in range(-max_shift, max_shift + 1):
                M = np.float32([[1, 0, dx],
                                [0, 1, dy]])
                shifted = cv2.warpAffine(
                    img,
                    M,
                    (img.shape[1], img.shape[0]),
                    flags=cv2.INTER_LINEAR,
                    borderMode=cv2.BORDER_CONSTANT,
                    borderValue=0,
                )
                denom = ref_norm * np.linalg.norm(shifted)
                if denom == 0:
                    continue
                score = np.sum(ref * shifted) / denom
                if score > best_score:
                    best_score = score
                    best_dx = dx
                    best_dy = dy
        # Keep the same convention as your original function
        shifts.append([-best_dx, best_dy])
    return xr.DataArray(
        np.asarray(shifts, dtype=np.float64).round(8),
        dims=["session", "shift_dim"],
        coords={
            "session": images.session,
            "shift_dim": ["height", "width"],
        },
        name="shifts",
    )

rois = A.max("unit_id").rename('rois')
shifts = compute_shifts(rois)
shifts_sorted=shifts.sortby("session")
for i, (session, shift) in enumerate(zip(shifts_sorted.session.values, shifts_sorted.values)):
    print(f"Session #{i}, {session}: {shift}")

Apply the shifts

In [111]:
try: 
    temps_sh = apply_transform(temps, shifts).compute().rename('temps_shifted')
    shiftds = xr.merge([temps, shifts, temps_sh])

    rois = A.max("unit_id").rename('rois')
    rois_sh = apply_transform(rois, shifts).compute().rename('rois_shifted')

    window = shiftds['temps_shifted'].isnull().sum('session')
    window, _ = xr.broadcast(window, shiftds['temps_shifted'])
except: 
    rois_sh = apply_transform(rois, shifts).compute().rename('rois_shifted')
    shiftds = xr.merge([rois, shifts, rois_sh])

    temps = minian_ds['max_proj'].rename('temps')
    temps_sh = apply_transform(temps, shifts).compute().rename('temps_shifted')

    window = shiftds['rois_shifted'].isnull().sum('session')
    window, _ = xr.broadcast(window, shiftds['rois_shifted'])

[########################################] | 100% Completed | 45.61 s
[########################################] | 100% Completed | 45.61 s


Visualize results across all sessions

In [ ]:
hv.output(size=int(output_size * 0.6))
opts_im = {'aspect': shiftds.sizes['width'] / shiftds.sizes['height'],'frame_width': 600, 'cmap': 'hot'}
hv_temps = hv.Dataset(temps_sh).to(hv.Image, ['width', 'height'])
hv_rois = hv.Dataset(rois_sh).to(hv.Image, ['width', 'height'])
hv_temps.opts(**opts_im).relabel('Image') + hv_rois.opts(**opts_im).relabel('ROIs')

In [ ]:
"""
c = minian_ds["C"].sel(session="2025_09_04_12_57_34")
plt.close('all')
plt.plot(c.sel(unit_id=138).values)
plt.plot(c.sel(unit_id=137).values)
plt.show()
mappings_meta_fill.loc[45] 
"""

## Change the shift manually /!\ only if needed

Option #1 : Reset shifts to 0 for all sessions

In [ ]:
shifts_modified = shifts.copy()
for i in np.arange(len(shifts_modified)):
    shifts_modified[i]=[0,0]

Option #2 : Change shift manually session by session

In [ ]:
shifts_modified = shifts.copy()
Session_to_shift='2025_08_13_13_55_31'
shifts_modified.loc[{'session': Session_to_shift}] = np.array([60, 20]) # Down_to_Up, Left_to_Right

Option #3: Shift one or multiple sessions using slide bars

In [ ]:
shifts_modified = shifts.copy(deep=True)
shifts_modified = shifts_modified.sortby("session")    
SessionTemplate = shifts_modified.session[15].item() # [0] the template session is the first one recorded
%matplotlib inline

def prepare_image(session):
    arr_temps = temps_sh.sel(session=session).copy() # choose to align with raw iamges
    arr_rois = rois_sh.sel(session=session).copy() # choose to align with rois
    arr_rois.values = np.flipud(np.rot90(arr_rois.values))
    arr_temps.values = np.flipud(np.rot90(arr_temps.values))
    return arr_temps, arr_rois

template_arr, template_rois = prepare_image(SessionTemplate)
template_img = hv.Image(template_arr).opts(width=800, height=800, cmap="Blues",)
template_rois_img = hv.Image(template_rois).opts(width=800, height=800, cmap="Blues",)
image_cache = {session: prepare_image(session) for session in shifts_modified.session.values}

def make_view(session, dx, dy, alpha):
    arr_temps, arr_rois = image_cache[session]
    shifted = arr_temps.shift(width=dy, height=dx)
    red_img = hv.Image(shifted).opts(width=800,height=800,cmap="Reds",alpha=alpha,title=(f"{session} | "f"dx={dx}, dy={dy} (red)"))
    return template_img * red_img

def make_view_2(session, dx, dy, alpha):
    arr_temps, arr_rois = image_cache[session]
    shifted = arr_rois.shift(width=dy, height=dx)
    red_img = hv.Image(shifted).opts(width=800,height=800,cmap="Reds",alpha=alpha,title=(f"{session} | "f"dx={dx}, dy={dy} (red)"))
    return template_rois_img * red_img

def make_combined_view(session, dx, dy, alpha):
    img1 = make_view(session, dx, dy, alpha)
    img2 = make_view_2(session, dx, dy, alpha)
    return (img1 + img2).opts(shared_axes=True)

def add_log(message):
    log_box.value += message + "\n"

session_widget = pn.widgets.Select(name="Selected session (red)",options=list(image_cache.keys()),value=SessionTemplate)
dx_widget = pn.widgets.IntSlider(name="Left → Right (dx)",start=-300,end=300,value=0)
dy_widget = pn.widgets.IntSlider(name="Down → Up (dy)",start=-300,end=300,value=0)
alpha_widget = pn.widgets.FloatSlider(name="Red opacity",start=0,end=1,step=0.05,value=0.5)
apply_button = pn.widgets.Button(name="Save shift for the selected session",button_type="success")
log_box = pn.widgets.TextAreaInput(name="Alignment log",value="",height=220,width=300,disabled=True)
prev_button = pn.widgets.Button(name="◀",button_type="default")
next_button = pn.widgets.Button(name="▶",button_type="default")
session_list = list(image_cache.keys())
title = pn.pane.HTML(f"""<div style="font-size: 20px;font-weight: bold;margin-bottom: 8px;">{SessionTemplate} (blue) vs selected session (red)</div>""")

def go_previous(event):
    current = session_widget.value
    idx = session_list.index(current)
    dx_widget.value = 0
    dy_widget.value = 0
    alpha_widget.value = 0.5
    if idx > 0:
        session_widget.value = session_list[idx - 1]

def go_next(event):
    current = session_widget.value
    idx = session_list.index(current)
    dx_widget.value = 0
    dy_widget.value = 0
    alpha_widget.value = 0.5
    if idx < len(session_list) - 1:
        session_widget.value = session_list[idx + 1]

def apply_shift(event):
    session = session_widget.value
    dy = dy_widget.value_throttled
    dx = dx_widget.value_throttled
    shifts_modified.loc[{"session": session}] = (shifts_modified.loc[{"session": session}].values + np.array([dy,dx]))
    add_log(f"{session}: dx = {dx}, dy = {dy}")
    dx_widget.value = 0
    dy_widget.value = 0
    alpha_widget.value = 0.5
    
def reset_sliders(event):
    dx_widget.value = 0
    dy_widget.value = 0
    alpha_widget.value = 0.5

session_widget.param.watch(reset_sliders,"value")
apply_button.on_click(apply_shift)
prev_button.on_click(go_previous)
next_button.on_click(go_next)

view = pn.bind(make_combined_view,session_widget,dx_widget.param.value,dy_widget.param.value,alpha_widget,)

layout = pn.Column(title, pn.Row(pn.Column(session_widget,pn.Row(prev_button, next_button),dx_widget,dy_widget,alpha_widget,apply_button,log_box,width=350),view))
layout

Apply modified shifts definetively

In [ ]:
shifts_modified = shifts_modified.sel(session=shifts.session) #reorder in the proper order (not by session date)
shifts = shifts_modified.copy(deep=True)
shifts_sorted = shifts.sortby("session")
for i, (session, shift_m) in enumerate(zip(shifts_sorted.session.values, shifts_sorted.values)):
    print(f"Session #{i}, {session}: {shift_m}")

temps_sh = apply_transform(temps, shifts).compute().rename('temps_shifted')
rois_sh = apply_transform(rois, shifts).compute().rename('rois_shifted')
shiftds = xr.merge([temps, shifts, temps_sh])

window = shiftds['temps_shifted'].isnull().sum('session')
window, _ = xr.broadcast(window, shiftds['temps_shifted'])

Visualize results across all sessions

In [ ]:
hv.output(size=int(output_size * 0.6))
opts_im = {'aspect': shiftds.sizes['width'] / shiftds.sizes['height'],'frame_width': 600, 'cmap': 'hot'}
hv_temps = hv.Dataset(temps_sh).to(hv.Image, ['width', 'height'])
hv_rois = hv.Dataset(rois_sh).to(hv.Image, ['width', 'height'])
hv_temps.opts(**opts_im).relabel('Image') + hv_rois.opts(**opts_im).relabel('ROIs')

## Compute mapping

Apply shifts and set window

In [ ]:
A_shifted = apply_transform(minian_ds['A'].chunk(dict(height=-1, width=-1)), shiftds['shifts'])

def set_window(wnd):
    return wnd == wnd.min()

window = xr.apply_ufunc(
    set_window,
    window,
    input_core_dims=[['height', 'width']],
    output_core_dims=[['height', 'width']],
    vectorize=True)

Calculate centroid distance

In [ ]:
param_dist = 5 # For LensConfig#1 : param_dist = 5, tile = (30, 30) // For LensConfig #2 : param_dist = 7, tile = (50, 50). # Default : param_dist = 5  
tiles = (30, 30) # Default : tile = (50, 50) 
cents = calculate_centroids(A_shifted, window)
try: 
    id_dims.remove("session")
except:
    print("'session' dimension already removed from id_dims")

# Remove dropped units (AB modified => works much better!)
cents_unittokeep = cents.copy()
for session, cell_nums in drop_units.items():
    cents_unittokeep = cents_unittokeep[~((cents_unittokeep['session'] == session) &(cents_unittokeep['unit_id'].isin(cell_nums)))]

# Compute regristration from kept cells only (AB modified => works much better!)
dist = calculate_centroid_distance(cents_unittokeep, index_dim = id_dims, tile = tiles) 
dist_ft = dist[dist['variable', 'distance'] < param_dist].copy()
dist_ft = group_by_session(dist_ft)

Generate mappings

In [ ]:
mappings = calculate_mapping(dist_ft)
mappings_meta = resolve_mapping(mappings)
mappings_meta_fill = fill_mapping(mappings_meta, cents_unittokeep)
print(len(mappings_meta_fill), "unique units kept for this mouse")
#mappings_meta_fill

## Check & save results

Plot results

In [ ]:
%matplotlib widget
n_sessions = temps_sh.sizes["session"]
temps_sh_sorted = temps_sh.sortby("session")
ncols = 6
nrows = math.ceil(n_sessions / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3*nrows),squeeze=False, sharex=True, sharey=True)
vmin, vmax = float(temps_sh_sorted.min()), float(temps_sh_sorted.max())
for i, ax in enumerate(axes.flat):
    if i >= n_sessions:
        ax.axis("off")
        continue
    session = temps_sh_sorted.session.values[i]
    temps_sh_sorted.isel(session=i).plot.imshow(ax=ax, vmin=vmin, vmax=vmax, add_colorbar=False)
    ax.set_title(str(session))
    ax.axis("off")
    cents_sess = cents_unittokeep[cents_unittokeep.session == session]
    ax.scatter(cents_sess.width, cents_sess.height, s=1, c="white")
    for _, r in cents_sess.iterrows():
        label = mappings_meta_fill.index[mappings_meta_fill[("session", session)] == float(r.unit_id)].tolist()
        ax.annotate(" " + str(label).replace("[", "").replace("]", ""),(r.width, r.height), xycoords="data", fontsize=7, color="white", annotation_clip=True)
        if not label:
            print(f'NO LABEL FOR UNIT#{int(r.unit_id)} SESSION {session} !!!! Position = {int(r.width)}, {int(r.height)}')
plt.tight_layout()
plt.show()

Save results

In [ ]:
mappings_meta_fill.to_pickle(os.path.join(dpath, "mappings_Oct10.pkl"))
mappings_meta_fill.to_csv(Path(dpath)/ f'mappings_Oct10.csv', index=False)  # cause problem to read pickle files on cluster 
cents.to_pickle(os.path.join(dpath, "cents_Oct10.pkl"))
shiftds.to_netcdf(os.path.join(dpath, "shiftds_Oct10.nc"))

## Edit cross-registration manually

Click on the wrong label, edit on the "Label" text box, press Enter. The new mapping is saved automatically! 

In [ ]:
# old with roi red on black background

%matplotlib widget

temps_sh_sorted = temps_sh.sortby("session")
rois_sh_sorted = rois_sh.sortby("session")
n_sessions = temps_sh.sizes["session"]
ncols = 6
nrows = math.ceil(n_sessions / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3*nrows), squeeze=False, sharex=True, sharey=True)
plt.subplots_adjust(bottom=0.08, wspace=0.05, hspace=0.15)

vmin, vmax = float(temps_sh_sorted.min()), float(temps_sh_sorted.max())
vmin, vmax = float(rois_sh_sorted.min()), float(rois_sh_sorted.max())

annotations = []
selected = [None]

for i, ax in enumerate(axes.flat):
    if i >= n_sessions:
        ax.axis("off")
        continue
    session = temps_sh_sorted.session.values[i]
    #temps_sh_sorted.isel(session=i).plot.imshow(ax=ax, vmin=vmin, vmax=vmax, add_colorbar=False)
    rois_sh_sorted.isel(session=i).plot.imshow(ax=ax, vmin=vmin, vmax=vmax, add_colorbar=False, cmap= 'copper')
    ax.set_title(str(session))
    ax.axis("off")
    cents_sess = cents_unittokeep[cents_unittokeep.session == session]
    ax.scatter(cents_sess.width, cents_sess.height, s=1, c="white")
    for _, r in cents_sess.iterrows():
        label = mappings_meta_fill.index[mappings_meta_fill[("session", session)] == float(r.unit_id)].tolist()
        ann = ax.annotate(" " + str(label).replace("[", "").replace("]", "").replace("'", ""), (r.width, r.height), xycoords="data", fontsize=7, color="white", annotation_clip=True, picker=True)
        ann.session, ann.unit_id = session, float(r.unit_id)
        annotations.append(ann)

box_ax = fig.add_axes([0.35, 0.01, 0.05, 0.04])
text_box = TextBox(box_ax, "Label: ")

def on_pick(event):
    if event.artist in annotations:
        selected[0] = event.artist
        text_box.set_val(event.artist.get_text().strip())

def on_submit(text):
    ann = selected[0]
    if ann is None or not text.strip():
        return
    session, unit_id = ann.session, ann.unit_id
    col = ("session", session)
    idx = mappings_meta_fill.index[mappings_meta_fill[col] == unit_id] # find old label (=index) 
    mappings_meta_fill.loc[idx, col] = np.nan # erase unit value (set to nan) at this index
    mappings_meta_fill.loc[int(text.strip()), col] = unit_id # set the unit value at the proper index
    ann.set_text(" " + text.strip())
    mappings_meta_fill.to_pickle(os.path.join(dpath, "mappings_Oct10_edited.pkl"))
    fig.canvas.draw_idle()

fig.canvas.mpl_connect("pick_event", on_pick)
text_box.on_submit(on_submit)
plt.tight_layout(rect=[0, 0.07, 1, 1])
plt.show()

In [ ]:
#new with bleu background and miniscope frame

%matplotlib widget

temps_sh_sorted = temps_sh.sortby("session")
rois_sh_sorted = rois_sh.sortby("session")
n_sessions = temps_sh.sizes["session"]
ncols = 6
nrows = math.ceil(n_sessions / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3*nrows), squeeze=False, sharex=True, sharey=True)
plt.subplots_adjust(bottom=0.08, wspace=0.05, hspace=0.15)

vmin, vmax = float(temps_sh_sorted.min()), float(temps_sh_sorted.max())
#vmin, vmax = float(rois_sh_sorted.min()), float(rois_sh_sorted.max())

annotations = []
selected = [None]

for i, ax in enumerate(axes.flat):
    if i >= n_sessions:
        ax.axis("off")
        continue
    session = temps_sh_sorted.session.values[i]
    temps_sh_sorted.isel(session=i).plot.imshow(ax=ax, vmin=vmin, vmax=vmax, add_colorbar=False)
    #rois_sh_sorted.isel(session=i).plot.imshow(ax=ax, vmin=vmin, vmax=vmax, add_colorbar=False, cmap= 'copper')
    ax.set_title(str(session))
    ax.axis("off")
    cents_sess = cents_unittokeep[cents_unittokeep.session == session]
    ax.scatter(cents_sess.width, cents_sess.height, s=1, c="white")
    for _, r in cents_sess.iterrows():
        label = mappings_meta_fill.index[mappings_meta_fill[("session", session)] == float(r.unit_id)].tolist()
        ann = ax.annotate(" " + str(label).replace("[", "").replace("]", "").replace("'", ""), (r.width, r.height), xycoords="data", fontsize=7, color="white", annotation_clip=True, picker=True)
        ann.session, ann.unit_id = session, float(r.unit_id)
        annotations.append(ann)

box_ax = fig.add_axes([0.35, 0.01, 0.05, 0.04])
text_box = TextBox(box_ax, "Label: ")

def on_pick(event):
    if event.artist in annotations:
        selected[0] = event.artist
        text_box.set_val(event.artist.get_text().strip())

def on_submit(text):
    ann = selected[0]
    if ann is None or not text.strip():
        return
    session, unit_id = ann.session, ann.unit_id
    col = ("session", session)
    idx = mappings_meta_fill.index[mappings_meta_fill[col] == unit_id] # find old label (=index) 
    mappings_meta_fill.loc[idx, col] = np.nan # erase unit value (set to nan) at this index
    mappings_meta_fill.loc[int(text.strip()), col] = unit_id # set the unit value at the proper index
    ann.set_text(" " + text.strip())
    mappings_meta_fill.to_pickle(os.path.join(dpath, "mappings_Oct10_edited.pkl"))
    fig.canvas.draw_idle()

fig.canvas.mpl_connect("pick_event", on_pick)
text_box.on_submit(on_submit)
plt.tight_layout(rect=[0, 0.07, 1, 1])
plt.show()

In [ ]:
print('->', sum(mappings_meta_fill['session'].isna().all(axis=1)), "units merged after editing")
df = mappings_meta_fill[~mappings_meta_fill['session'].isna().all(axis=1)]
print(len(df), "unique units kept for this mouse")

Save edited mapping

In [ ]:
mappings_meta_fill.to_pickle(os.path.join(dpath, "mappings_Oct10_edited.pkl"))

## Plot all sessions mapping on one image

merged sessions

In [ ]:
%matplotlib widget
temps_sh_sorted = temps_sh.sortby("session")
fig, ax = plt.subplots(figsize=(12, 10))
vmin, vmax = float(temps_sh_sorted.min()), float(temps_sh_sorted.max())
temps_sh_sorted.isel(session=0).plot.imshow(ax=ax, vmin=vmin, vmax=vmax, add_colorbar=False)
plotted = set()
for session in temps_sh_sorted.session.values:
    cents_sess = cents_unittokeep[cents_unittokeep.session == session]
    for _, r in cents_sess.iterrows():
        label = mappings_meta_fill.index[mappings_meta_fill[("session", session)] == float(r.unit_id)].tolist()
        if label and label[0] not in plotted:
            ax.scatter(r.width, r.height, s=1, c="white")
            ax.annotate(f" {label[0]}", (r.width, r.height),fontsize=8, color="white")
            plotted.add(label[0])
        if not label: 
            print(f'NO LABEL FOR UNIT#{int(r.unit_id)} SESSION {session} !!!! Position = {int(r.width)}, {int(r.height)}')
            ax.scatter(r.width, r.height, s=10, c="red")
fig.subplots_adjust(right=0.78)
plt.show()

with one color = one session

In [ ]:
%matplotlib widget
temps_sh_sorted = temps_sh.sortby("session")
fig, ax = plt.subplots(figsize=(12,10))

vmin, vmax = float(temps_sh_sorted.min()), float(temps_sh_sorted.max())
temps_sh_sorted.isel(session=0).plot.imshow(ax=ax, vmin=vmin, vmax=vmax, add_colorbar=False)

dsessions = cents_unittokeep["session"].unique()
colors = plt.cm.hsv(np.linspace(0, 1, len(dsessions)))
color_map = dict(zip(dsessions, colors))

for session in temps_sh_sorted.session.values:
    cents_sess = cents_unittokeep[cents_unittokeep.session == session]
    color = color_map[session]
    #color= 'white'
    ax.scatter(cents_sess.width, cents_sess.height, s=1, c=[color])
    for _, r in cents_sess.iterrows():
        label = mappings_meta_fill.index[mappings_meta_fill[("session", session)] == float(r.unit_id)].tolist()
        ax.annotate(f" {label[0]}" if label else "",(r.width, r.height),fontsize=8, color=color, annotation_clip=True)
        if not label: 
            print(f'NO LABEL FOR UNIT#{int(r.unit_id)} SESSION {session} !!!! Position = {int(r.width)}, {int(r.height)}')
            ax.scatter(r.width, r.height, s=10, c="black")
ax.legend( handles=[ Line2D([0], [0], marker="o", color="w", markerfacecolor=color, markersize=7, label=str(s)) for s, color in color_map.items() ], title="session", bbox_to_anchor=(1.02, 1), loc="upper left", ) 
fig.subplots_adjust(right=0.78) 
plt.show()

all sessions

In [ ]:
%matplotlib widget
temps_sh_sorted = temps_sh.sortby("session")
fig, ax = plt.subplots(figsize=(12,10))

vmin, vmax = float(temps_sh_sorted.min()), float(temps_sh_sorted.max())
temps_sh_sorted.isel(session=0).plot.imshow(ax=ax, vmin=vmin, vmax=vmax, add_colorbar=False)

dsessions = cents_unittokeep["session"].unique()
colors = plt.cm.hsv(np.linspace(0, 1, len(dsessions)))
color_map = dict(zip(dsessions, colors))

for session in temps_sh_sorted.session.values:
    cents_sess = cents_unittokeep[cents_unittokeep.session == session]
    color = color_map[session]
    color= 'white'
    ax.scatter(cents_sess.width, cents_sess.height, s=1, c=[color])
    for _, r in cents_sess.iterrows():
        label = mappings_meta_fill.index[mappings_meta_fill[("session", session)] == float(r.unit_id)].tolist()
        ax.annotate(f" {label[0]}" if label else "",(r.width, r.height),fontsize=8, color='white', annotation_clip=True)
        if not label: 
            print(f'NO LABEL FOR UNIT#{int(r.unit_id)} SESSION {session} !!!! Position = {int(r.width)}, {int(r.height)}')
            ax.scatter(r.width, r.height, s=10, c="red")
fig.subplots_adjust(right=0.78) 
plt.show()